# Session 2: Deployment and Scaling (Instructor Notebook)

In this session students learn how to take McKinsey consulting AI applications from prototype to production. We cover API design for consulting services (engagement assistant, knowledge search, client briefing generator), caching strategies for recurring consulting queries, monitoring for partner-review quality metrics, model routing by consulting complexity, and cost tracking per engagement.

> **Instructor Note:** This notebook contains all 5 demos (identical to student) plus fully solved versions of all 4 tasks with approach explanations. All examples use McKinsey consulting scenarios.

## Learning Objectives

By the end of this session, students will be able to:

1. **Design** a FastAPI-style service for McKinsey's consulting AI (engagement assistant, knowledge search, client briefing APIs)
2. **Implement** semantic caching to reduce costs on recurring consulting queries (market research, industry analyses)
3. **Add** monitoring and structured logging that tracks consulting-relevant metrics (response quality, framework coverage, analysis accuracy)
4. **Build** a model router that selects models based on consulting complexity (fact lookup vs. strategy analysis)
5. **Track** token usage and costs modeled around consulting AI economics (cost per engagement, tokens per client deliverable)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
import hashlib
import logging
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Designing a McKinsey Consulting AI API Service

Production consulting AI applications need a clean API layer. We design a service class that wraps the LLM with request validation, error handling, and structured responses -- the same patterns you would use to build McKinsey's Engagement Assistant API or Knowledge Search API in a FastAPI endpoint.

In [ ]:
# Demo 1 - McKinsey Consulting AI API Service Design

@dataclass
class ConsultingAIRequest:
    """Validated request for the McKinsey Consulting AI service."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.7
    request_id: str = ""
    engagement_id: str = "ENG-DEFAULT"  # McKinsey engagement identifier
    service_type: str = "knowledge_search"  # knowledge_search | engagement_assistant | briefing_generator
    
    def __post_init__(self):
        if not self.request_id:
            self.request_id = hashlib.md5(f"{self.prompt}{time.time()}".encode()).hexdigest()[:8]
        if self.max_tokens < 1 or self.max_tokens > 4096:
            raise ValueError("max_tokens must be between 1 and 4096")
        if self.temperature < 0 or self.temperature > 2:
            raise ValueError("temperature must be between 0 and 2")
        valid_services = ["knowledge_search", "engagement_assistant", "briefing_generator"]
        if self.service_type not in valid_services:
            raise ValueError(f"service_type must be one of {valid_services}")

@dataclass
class ConsultingAIResponse:
    """Structured response from the McKinsey Consulting AI service."""
    content: str
    model: str
    request_id: str
    tokens_used: int
    latency_ms: float
    engagement_id: str = ""
    service_type: str = ""
    status: str = "success"
    error: Optional[str] = None

class McKinseyAIService:
    """Production consulting AI service with validation and error handling.
    
    Supports three API endpoints:
    - Knowledge Search: Query McKinsey's internal knowledge base
    - Engagement Assistant: Help consultants with active engagements
    - Briefing Generator: Generate client-ready briefing documents
    """
    
    SYSTEM_PROMPTS = {
        "knowledge_search": "You are McKinsey's Knowledge Search AI. Provide concise, framework-driven answers drawing on consulting best practices, industry analyses, and strategic frameworks (Porter's Five Forces, 7S, MECE, etc.).",
        "engagement_assistant": "You are McKinsey's Engagement Assistant AI. Help consultants with active client engagements by providing structured analysis, workstream planning, and deliverable outlines.",
        "briefing_generator": "You are McKinsey's Client Briefing Generator. Create executive-ready briefing content that is structured, data-driven, and actionable for C-suite audiences."
    }
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def invoke(self, request: ConsultingAIRequest) -> ConsultingAIResponse:
        start = time.time()
        system_prompt = self.SYSTEM_PROMPTS.get(request.service_type, self.SYSTEM_PROMPTS["knowledge_search"])
        try:
            response = self.client.chat.completions.create(
                model=request.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": request.prompt}
                ],
                max_tokens=request.max_tokens,
                temperature=request.temperature
            )
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content=response.choices[0].message.content,
                model=request.model,
                request_id=request.request_id,
                tokens_used=response.usage.total_tokens,
                latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type
            )
        except Exception as e:
            latency = (time.time() - start) * 1000
            return ConsultingAIResponse(
                content="", model=request.model,
                request_id=request.request_id,
                tokens_used=0, latency_ms=round(latency, 2),
                engagement_id=request.engagement_id,
                service_type=request.service_type,
                status="error", error=str(e)
            )

# Test the McKinsey Consulting AI Service
service = McKinseyAIService()

# Knowledge Search API call
req = ConsultingAIRequest(
    prompt="What are the key elements of a McKinsey market entry strategy for a healthcare client?",
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150")), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")),
    engagement_id="ENG-2024-0892",
    service_type="knowledge_search"
)
resp = service.invoke(req)

print(f"Request ID:    {resp.request_id}")
print(f"Engagement:    {resp.engagement_id}")
print(f"Service Type:  {resp.service_type}")
print(f"Status:        {resp.status}")
print(f"Model:         {resp.model}")
print(f"Tokens:        {resp.tokens_used}")
print(f"Latency:       {resp.latency_ms}ms")
print(f"Response:      {resp.content}")

### Demo 2: Semantic Caching for Consulting Queries

LLM calls are expensive and slow. At a firm like McKinsey, consultants across engagements frequently ask similar market research questions and recurring industry analyses. Semantic caching stores previous responses and returns them for similar (not just identical) queries -- reducing costs by 30-70% when multiple teams research overlapping industries or frameworks.

In [ ]:
# Demo 2 - Semantic Caching for McKinsey Consulting Queries

class SemanticCache:
    """Cache that matches by semantic similarity, not exact string match.
    
    Designed for McKinsey consulting queries where different consultants
    often ask similar market research and industry analysis questions.
    """
    
    def __init__(self, similarity_threshold=0.92):
        self.client = openai.OpenAI()
        self.cache = []
        self.threshold = similarity_threshold
        self.hits = 0
        self.misses = 0
    
    def _embed(self, text):
        return self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get(self, query):
        query_emb = self._embed(query)
        best_sim, best_response = 0, None
        for cached_emb, cached_query, cached_response in self.cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = (cached_query, cached_response)
        if best_sim >= self.threshold:
            self.hits += 1
            return {"hit": True, "similarity": best_sim,
                    "cached_query": best_response[0], "response": best_response[1]}
        self.misses += 1
        return {"hit": False, "similarity": best_sim}
    
    def put(self, query, response):
        emb = self._embed(query)
        self.cache.append((emb, query, response))

# Test with McKinsey consulting queries
cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# First query -- a market research question (cache miss)
q1 = "What are the key drivers of digital transformation in the healthcare industry?"
result = cache.get(q1)
print(f"Query: {q1}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")

answer = llm.invoke([HumanMessage(content=q1)]).content
cache.put(q1, answer)
print(f"Cached response: {answer[:100]}...")

# Similar consulting query -- should hit cache (different phrasing, same research question)
q2 = "What is driving digital transformation in healthcare?"
result = cache.get(q2)
print(f"\nQuery: {q2}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
if result['hit']:
    print(f"Matched to: {result['cached_query']}")

# Different industry analysis -- should miss (unrelated topic)
q3 = "What are the main cost reduction levers in automotive manufacturing?"
result = cache.get(q3)
print(f"\nQuery: {q3}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
print(f"\nStats: {cache.hits} hits, {cache.misses} misses")

### Demo 3: Monitoring and Structured Logging for Consulting AI

In production, McKinsey partners and engagement managers need visibility into AI performance. Structured logging captures every request, response quality metrics, token count, latency, and errors -- enabling partner review of AI-generated analyses and tracking framework coverage across engagements.

In [ ]:
# Demo 3 - Monitoring and Structured Logging for McKinsey Consulting AI

class ConsultingAIMonitor:
    """Monitors McKinsey consulting AI usage with structured logging and metrics.
    
    Tracks consulting-relevant metrics: response quality for partner review,
    analysis accuracy, framework coverage, and per-engagement performance.
    """
    
    def __init__(self):
        self.logs = []
        self.metrics = defaultdict(list)
    
    def log_request(self, request_id, model, prompt, response, tokens, latency_ms,
                    engagement_id="", service_type="", status="success"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id,
            "engagement_id": engagement_id,
            "service_type": service_type,
            "model": model,
            "prompt_preview": prompt[:100],
            "response_preview": response[:100] if response else "",
            "tokens": tokens,
            "latency_ms": latency_ms,
            "status": status
        }
        self.logs.append(entry)
        self.metrics["latency"].append(latency_ms)
        self.metrics["tokens"].append(tokens)
        self.metrics["models"].append(model)
        self.metrics["service_types"].append(service_type)
        return entry
    
    def get_summary(self):
        if not self.logs:
            return "No requests logged."
        latencies = self.metrics["latency"]
        tokens = self.metrics["tokens"]
        models = self.metrics["models"]
        service_types = self.metrics["service_types"]
        return {
            "total_requests": len(self.logs),
            "avg_latency_ms": round(np.mean(latencies), 2),
            "p95_latency_ms": round(np.percentile(latencies, 95), 2),
            "total_tokens": sum(tokens),
            "avg_tokens_per_query": round(np.mean(tokens), 2),
            "model_distribution": {m: models.count(m) for m in set(models)},
            "service_type_distribution": {s: service_types.count(s) for s in set(service_types) if s},
            "error_rate": sum(1 for l in self.logs if l["status"] != "success") / len(self.logs),
            "partner_review_ready": sum(1 for l in self.logs if l["status"] == "success")
        }

# Test with McKinsey consulting queries
monitor = ConsultingAIMonitor()
client = openai.OpenAI()

# Simulate consulting AI queries across different service types
consulting_queries = [
    ("What is McKinsey's 7S framework?", "knowledge_search", "ENG-2024-0101"),
    ("Summarize the competitive landscape in European fintech.", "engagement_assistant", "ENG-2024-0205"),
    ("What are the top 3 cost reduction levers for a telecom client?", "knowledge_search", "ENG-2024-0312"),
    ("Draft an executive summary for the digital transformation engagement.", "briefing_generator", "ENG-2024-0205"),
    ("Compare market entry strategies: greenfield vs. acquisition vs. JV.", "knowledge_search", "ENG-2024-0101"),
]

for prompt, svc_type, eng_id in consulting_queries:
    start = time.time()
    resp = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "80")))
    latency = (time.time() - start) * 1000
    entry = monitor.log_request(
        request_id=hashlib.md5(prompt.encode()).hexdigest()[:8], model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        prompt=prompt, response=resp.choices[0].message.content,
        tokens=resp.usage.total_tokens, latency_ms=round(latency, 2),
        engagement_id=eng_id, service_type=svc_type
    )
    print(f"[{entry['request_id']}] [{entry['service_type']:22s}] {entry['prompt_preview'][:45]:45s} | {entry['tokens']}tok | {entry['latency_ms']}ms")

print("\n--- Consulting AI Performance Summary ---")
for k, v in monitor.get_summary().items():
    print(f"  {k}: {v}")

### Demo 4: Model Routing by Consulting Complexity

Not every consulting query needs GPT-4o. A model router classifies query complexity and routes accordingly: simple fact lookups (framework definitions, industry stats) go to gpt-4o-mini, while complex strategy analyses (market entry evaluation, M&A synergy modeling, competitive response planning) go to gpt-4o -- cutting costs by 40-60% with minimal quality loss on routine queries.

In [ ]:
# Demo 4 - Model Routing by McKinsey Consulting Complexity

class ConsultingModelRouter:
    """Routes consulting queries to appropriate models based on complexity.
    
    Routing logic reflects McKinsey engagement patterns:
    - simple: Framework definitions, industry stats, quick fact lookups -> gpt-4o-mini
    - medium: Industry comparisons, trend summaries, workstream planning -> gpt-4o-mini
    - complex: Multi-dimensional strategy analysis, M&A evaluation, competitive response -> gpt-4o
    """
    
    MODEL_TIERS = {
        "simple": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "medium": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "complex": {"model": "gpt-4o", "cost_per_1k": 0.005}
    }
    
    def __init__(self):
        self.classifier = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
    
    def classify_complexity(self, query):
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this McKinsey consulting query's complexity. Return ONLY one word:
- simple: factual lookups, framework definitions, industry statistics, quick answers
- medium: industry comparisons, trend summaries, moderate analysis, workstream planning
- complex: multi-step strategy analysis, M&A evaluation, market entry planning, competitive response design"""),
            HumanMessage(content=query)
        ])
        complexity = response.content.strip().lower()
        return complexity if complexity in self.MODEL_TIERS else "medium"
    
    def route(self, query):
        complexity = self.classify_complexity(query)
        tier = self.MODEL_TIERS[complexity]
        return {"complexity": complexity, "model": tier["model"], "cost_per_1k": tier["cost_per_1k"]}

# Test with McKinsey consulting queries of varying complexity
router = ConsultingModelRouter()
consulting_queries = [
    "What does MECE stand for?",
    "Compare the competitive landscape of European vs. US fintech markets across regulation, adoption, and profitability.",
    "Design a post-merger integration plan for a $2B healthcare acquisition including synergy capture, org design, and Day 1 readiness.",
    "What is Porter's Five Forces framework?",
    "Develop a market entry strategy for a Fortune 500 retailer entering Southeast Asian e-commerce with build, buy, and partner options."
]

for q in consulting_queries:
    result = router.route(q)
    print(f"[{result['complexity']:7s}] -> {result['model']:12s} (${result['cost_per_1k']}/1k) | {q[:70]}")

### Demo 5: Consulting AI Cost Tracking per Engagement

Tracking costs is critical for McKinsey's AI economics. This demo builds a cost tracker that monitors token usage per model, calculates cost per engagement analysis, tracks tokens per client deliverable, and provides spending alerts -- enabling the firm to model AI costs as part of engagement budgeting.

In [ ]:
# Demo 5 - McKinsey Consulting AI Cost Tracking per Engagement

class ConsultingCostTracker:
    """Tracks token usage and costs across models for McKinsey engagements.
    
    Models consulting AI economics: cost per engagement analysis,
    token usage per client deliverable, and budget alerts per engagement.
    """
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "text-embedding-3-small": {"input": 0.00002, "output": 0}
    }
    
    def __init__(self, budget_limit=1.0):
        self.budget_limit = budget_limit
        self.records = []
    
    def record(self, model, input_tokens, output_tokens, request_id="", engagement_id=""):
        pricing = self.PRICING.get(model, {"input": 0.001, "output": 0.002})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        entry = {"timestamp": datetime.now().isoformat(), "model": model,
                 "input_tokens": input_tokens, "output_tokens": output_tokens,
                 "cost": cost, "request_id": request_id, "engagement_id": engagement_id}
        self.records.append(entry)
        total = sum(r["cost"] for r in self.records)
        if total > self.budget_limit * 0.8:
            print(f"  WARNING: Engagement AI budget at {total/self.budget_limit*100:.1f}%")
        return entry
    
    def get_report(self):
        total_cost = sum(r["cost"] for r in self.records)
        by_model = defaultdict(lambda: {"requests": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0})
        by_engagement = defaultdict(lambda: {"requests": 0, "cost": 0})
        for r in self.records:
            by_model[r["model"]]["requests"] += 1
            by_model[r["model"]]["input_tokens"] += r["input_tokens"]
            by_model[r["model"]]["output_tokens"] += r["output_tokens"]
            by_model[r["model"]]["cost"] += r["cost"]
            if r.get("engagement_id"):
                by_engagement[r["engagement_id"]]["requests"] += 1
                by_engagement[r["engagement_id"]]["cost"] += r["cost"]
        return {"total_cost": total_cost, "total_requests": len(self.records),
                "budget_remaining": self.budget_limit - total_cost,
                "cost_per_analysis": total_cost / len(self.records) if self.records else 0,
                "by_model": dict(by_model), "by_engagement": dict(by_engagement)}

# Test with McKinsey consulting AI queries across engagements
tracker = ConsultingCostTracker(budget_limit=0.05)
client = openai.OpenAI()

consulting_queries = [
    ("gpt-4o-mini", "What is McKinsey's 7S framework?", "ENG-2024-0101"),
    ("gpt-4o-mini", "List the top 3 players in European fintech.", "ENG-2024-0205"),
    ("gpt-4o", "Design a post-merger integration roadmap for a healthcare acquisition covering synergy capture and org restructuring.", "ENG-2024-0312"),
    ("gpt-4o-mini", "What are the main drivers of customer churn in telecom?", "ENG-2024-0205"),
]

for model, prompt, eng_id in consulting_queries:
    resp = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100")))
    entry = tracker.record(model, resp.usage.prompt_tokens, resp.usage.completion_tokens, engagement_id=eng_id)
    print(f"[{model:12s}] [{eng_id}] {prompt[:45]:45s} | ${entry['cost']:.6f}")

print("\n--- Consulting AI Cost Report ---")
report = tracker.get_report()
print(f"Total cost:           ${report['total_cost']:.6f}")
print(f"Cost per analysis:    ${report['cost_per_analysis']:.6f}")
print(f"Budget remaining:     ${report['budget_remaining']:.6f}")
print(f"\nBy Model:")
for model, stats in report['by_model'].items():
    print(f"  {model}: {stats['requests']} reqs, ${stats['cost']:.6f}")
print(f"\nBy Engagement:")
for eng, stats in report['by_engagement'].items():
    print(f"  {eng}: {stats['requests']} queries, ${stats['cost']:.6f}")

---
## Tasks (Solved)

### Task 1: Build a Rate-Limited Consulting AI Service

Create an LLM service wrapper that enforces rate limits -- both requests-per-minute and tokens-per-minute -- to prevent runaway costs during peak consulting seasons (e.g., M&A waves, year-end strategy reviews) when many consultants are using the AI simultaneously.

> **Approach:** We maintain a list of (timestamp, token_count) entries. Before each request, we prune entries older than 60 seconds, then check if adding a new request would exceed either the RPM or TPM limit. If so, we return a rate-limit error instead of calling the API. This protects engagement budgets during high-demand periods.

In [ ]:
# Task 1 — SOLUTION: Rate-Limited McKinsey Consulting AI Service

class RateLimitedConsultingAI:
    """Rate-limited consulting AI service to protect engagement budgets
    during peak seasons (M&A waves, year-end strategy reviews).
    """
    
    def __init__(self, rpm_limit=10, tpm_limit=5000):
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.client = openai.OpenAI()
        self.request_log = []  # List of (timestamp, token_count)
    
    def _prune_old_entries(self):
        """Remove entries older than 60 seconds."""
        cutoff = time.time() - 60
        self.request_log = [(ts, tok) for ts, tok in self.request_log if ts > cutoff]
    
    def _check_limits(self):
        """Check if we're within rate limits."""
        self._prune_old_entries()
        current_rpm = len(self.request_log)
        current_tpm = sum(tok for _, tok in self.request_log)
        
        if current_rpm >= self.rpm_limit:
            return False, f"RPM limit reached ({current_rpm}/{self.rpm_limit})"
        if current_tpm >= self.tpm_limit:
            return False, f"TPM limit reached ({current_tpm}/{self.tpm_limit})"
        return True, "OK"
    
    def invoke(self, prompt, max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))):
        """Make a rate-limited consulting AI call."""
        allowed, reason = self._check_limits()
        if not allowed:
            return {"status": "rate_limited", "reason": reason, "content": None}
        
        try:
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=[
                    {"role": "system", "content": "You are McKinsey's consulting AI assistant. Provide structured, framework-driven analysis."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens
            )
            tokens = response.usage.total_tokens
            self.request_log.append((time.time(), tokens))
            return {
                "status": "success",
                "content": response.choices[0].message.content,
                "tokens": tokens
            }
        except Exception as e:
            return {"status": "error", "reason": str(e), "content": None}


# Test with a low limit to simulate peak M&A season demand
rl_llm = RateLimitedConsultingAI(rpm_limit=3, tpm_limit=1000)

consulting_prompts = [
    "What is McKinsey's 7S framework? Answer in one sentence.",
    "What are the key drivers of M&A synergy capture? Answer in one sentence.",
    "List the top 3 market entry strategies. Answer in one sentence.",
    "What is a profit pool analysis? Answer in one sentence.",
    "Define MECE in one sentence."
]

for i, prompt in enumerate(consulting_prompts):
    result = rl_llm.invoke(prompt)
    if result['status'] == 'success':
        print(f"Request {i}: {result['status']} ({result['tokens']} tokens) - {result['content'][:60]}...")
    else:
        print(f"Request {i}: {result['status']} - {result.get('reason', '')}")

### Task 2: Implement a Streaming Client Briefing Generator

Build a streaming response handler for McKinsey's Client Briefing Generator that processes tokens as they arrive, tracks partial results, and provides real-time latency metrics -- critical for partners who need to see briefing content generating in real time during engagement reviews.

> **Approach:** We use OpenAI's streaming API (`stream=True`) and iterate over chunks. We track the timestamp of the first chunk (TTFT), count chunks as they arrive, and compute tokens-per-second from the total time and chunk count.

In [ ]:
# Task 2 — SOLUTION: Streaming Client Briefing Generator

class ConsultingStreamingHandler:
    """Streaming handler for McKinsey's Client Briefing Generator.
    
    Enables real-time briefing generation during partner reviews,
    with time-to-first-token and throughput metrics.
    """
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def stream(self, prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))):
        """Stream a consulting briefing response and track metrics."""
        start_time = time.time()
        first_token_time = None
        chunks = []
        token_count = 0
        
        response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are McKinsey's Client Briefing Generator. Create executive-ready briefing content that is structured, data-driven, and actionable."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens,
            stream=True
        )
        
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                if first_token_time is None:
                    first_token_time = time.time()
                chunks.append(content)
                token_count += 1
        
        end_time = time.time()
        total_time = end_time - start_time
        ttft = (first_token_time - start_time) * 1000 if first_token_time else 0
        tps = token_count / total_time if total_time > 0 else 0
        
        return {
            "content": "".join(chunks),
            "token_count": token_count,
            "ttft_ms": round(ttft, 2),
            "total_time_ms": round(total_time * 1000, 2),
            "tokens_per_sec": round(tps, 1)
        }


# Test with McKinsey consulting briefing requests
handler = ConsultingStreamingHandler()

briefing_prompts = [
    "Draft a 3-sentence executive summary for a telecom cost reduction engagement that achieved 15% OpEx savings.",
    "Summarize the key findings from a market entry analysis for a pharmaceutical company entering Southeast Asia."
]

for prompt in briefing_prompts:
    result = handler.stream(prompt)
    print(f"Briefing Request: {prompt[:70]}...")
    print(f"  TTFT: {result['ttft_ms']:.0f}ms | Total: {result['total_time_ms']:.0f}ms | Tokens/sec: {result['tokens_per_sec']}")
    print(f"  Briefing Content: {result['content'][:120]}...\n")

### Task 3: Build a Multi-Tier Caching System for Consulting Knowledge

Build a caching system with two tiers: exact-match (fast, hash-based) and semantic-match (slower, embedding-based). This is essential for McKinsey where consultants across engagements frequently ask identical or semantically similar industry research questions -- e.g., multiple teams researching "healthcare M&A trends" in slightly different phrasings.

> **Approach:** We use a dict keyed by prompt hash for O(1) exact matches, and an embedding-based list for semantic matches. The query method checks exact first, then semantic, then falls through to the LLM. Each tier tracks its own hit rate.

In [ ]:
# Task 3 — SOLUTION: Multi-Tier Caching for McKinsey Consulting Knowledge

class ConsultingTieredCache:
    """Multi-tier cache for McKinsey consulting knowledge queries.
    
    Tier 1: Exact match (hash-based, O(1)) -- same consultant re-asking
    Tier 2: Semantic match (embedding similarity) -- different consultants, similar research
    Tier 3: LLM call (cache miss) -- novel query
    """
    
    def __init__(self, semantic_threshold=0.92):
        self.exact_cache = {}  # hash -> response
        self.semantic_cache = []  # (embedding, query, response)
        self.threshold = semantic_threshold
        self.embed_client = openai.OpenAI()
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0}
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _embed(self, text):
        return self.embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def query(self, prompt):
        # Tier 1: Exact match
        key = self._hash(prompt)
        if key in self.exact_cache:
            self.stats["exact_hits"] += 1
            return {"content": self.exact_cache[key], "tier": "exact", "cached": True}
        
        # Tier 2: Semantic match
        query_emb = self._embed(prompt)
        best_sim, best_response = 0, None
        for cached_emb, cached_query, cached_response in self.semantic_cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = cached_response
        
        if best_sim >= self.threshold:
            self.stats["semantic_hits"] += 1
            # Also add to exact cache for future exact matches
            self.exact_cache[key] = best_response
            return {"content": best_response, "tier": "semantic", "similarity": best_sim, "cached": True}
        
        # Tier 3: LLM call
        self.stats["misses"] += 1
        response = self.llm.invoke([
            SystemMessage(content="You are McKinsey's knowledge search AI. Provide concise, framework-driven consulting insights."),
            HumanMessage(content=prompt)
        ]).content
        
        # Store in both caches
        self.exact_cache[key] = response
        self.semantic_cache.append((query_emb, prompt, response))
        
        return {"content": response, "tier": "llm", "cached": False}


# Test with McKinsey consulting knowledge queries
cache = ConsultingTieredCache(semantic_threshold=0.90)

test_queries = [
    "What are the key trends in healthcare M&A?",                  # LLM call (first time)
    "What are the key trends in healthcare M&A?",                  # Exact match (same consultant re-asking)
    "What is driving mergers and acquisitions in the healthcare sector?",  # Semantic match (different consultant, same topic)
    "What are the main cost reduction levers in automotive manufacturing?",  # LLM call (different topic)
    "What are the main cost reduction levers in automotive manufacturing?",  # Exact match
]

for q in test_queries:
    result = cache.query(q)
    tier_info = f"tier={result['tier']}"
    if 'similarity' in result:
        tier_info += f" sim={result['similarity']:.3f}"
    print(f"[{tier_info:25s}] {q[:50]:50s} | {result['content'][:50]}...")

print(f"\nConsulting Knowledge Cache Stats: {cache.stats}")

### Task 4: Create a Full Production McKinsey Consulting AI Gateway

Combine everything into a production gateway for McKinsey's consulting AI platform: model routing by consulting complexity, caching for recurring industry queries, rate limiting for peak engagement seasons, monitoring for partner review, and cost tracking per engagement.

> **Approach:** We compose the components from previous demos/tasks into a single gateway class. The pipeline is: classify consulting complexity -> check cache -> check rate limit -> call LLM -> cache response -> log & track cost per engagement. The dashboard method aggregates metrics for firm-wide AI performance review.

In [ ]:
# Task 4 — SOLUTION: Full Production McKinsey Consulting AI Gateway

class McKinseyAIGateway:
    """Production gateway for McKinsey's consulting AI platform.
    
    Combines model routing, caching, rate limiting, monitoring, and
    cost tracking -- designed for multi-engagement support, cross-industry
    knowledge, and peak season scalability (M&A waves, year-end reviews).
    """
    
    MODEL_TIERS = {
        "simple": "gpt-4o-mini",   # Framework definitions, industry stats
        "medium": "gpt-4o-mini",   # Comparisons, trend summaries
        "complex": "gpt-4o"        # Strategy analysis, M&A evaluation
    }
    
    def __init__(self, rpm_limit=20, tpm_limit=10000, budget=1.0):
        self.client = openai.OpenAI()
        self.classifier = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        
        # Cache for recurring consulting queries
        self.cache = {}  # hash -> (response, model)
        
        # Rate limiter for peak season protection
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.request_log = []
        
        # Monitoring for partner review
        self.logs = []
        
        # Cost tracking per engagement
        self.cost_tracker = ConsultingCostTracker(budget_limit=budget)
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _classify(self, query):
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this McKinsey consulting query complexity. Return ONLY: simple, medium, or complex.
- simple: framework definitions, industry stats, factual lookups
- medium: comparisons, trend summaries, workstream planning
- complex: strategy analysis, M&A evaluation, market entry design"""),
            HumanMessage(content=query)
        ])
        c = response.content.strip().lower()
        return c if c in self.MODEL_TIERS else "medium"
    
    def _check_rate_limit(self):
        cutoff = time.time() - 60
        self.request_log = [(ts, tok) for ts, tok in self.request_log if ts > cutoff]
        if len(self.request_log) >= self.rpm_limit:
            return False, "RPM limit exceeded -- peak consulting demand"
        if sum(tok for _, tok in self.request_log) >= self.tpm_limit:
            return False, "TPM limit exceeded -- engagement budget protection"
        return True, "OK"
    
    def query(self, prompt, engagement_id="ENG-DEFAULT"):
        start = time.time()
        
        # Step 1: Route by consulting complexity
        complexity = self._classify(prompt)
        model = self.MODEL_TIERS[complexity]
        
        # Step 2: Check cache (recurring industry queries)
        key = self._hash(prompt)
        if key in self.cache:
            cached_resp, cached_model = self.cache[key]
            latency = (time.time() - start) * 1000
            return {"content": cached_resp, "model": cached_model,
                    "cached": True, "complexity": complexity,
                    "latency_ms": round(latency, 2), "cost": 0.0,
                    "engagement_id": engagement_id}
        
        # Step 3: Check rate limit (peak season protection)
        allowed, reason = self._check_rate_limit()
        if not allowed:
            return {"content": None, "model": model, "cached": False,
                    "complexity": complexity, "status": "rate_limited", "reason": reason}
        
        # Step 4: Call LLM with consulting system prompt
        response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are McKinsey's consulting AI. Provide structured, framework-driven, executive-ready analysis."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
        )
        content = response.choices[0].message.content
        tokens = response.usage.total_tokens
        latency = (time.time() - start) * 1000
        
        # Step 5: Cache, log, track cost per engagement
        self.cache[key] = (content, model)
        self.request_log.append((time.time(), tokens))
        cost_entry = self.cost_tracker.record(model, response.usage.prompt_tokens,
                                               response.usage.completion_tokens,
                                               engagement_id=engagement_id)
        self.logs.append({"timestamp": datetime.now().isoformat(), "prompt": prompt[:80],
                          "model": model, "tokens": tokens, "latency_ms": round(latency, 2),
                          "engagement_id": engagement_id, "complexity": complexity})
        
        return {"content": content, "model": model, "cached": False,
                "complexity": complexity, "latency_ms": round(latency, 2),
                "cost": cost_entry["cost"], "tokens": tokens,
                "engagement_id": engagement_id}
    
    def get_dashboard(self):
        """Firm-wide consulting AI performance dashboard."""
        cost_report = self.cost_tracker.get_report()
        return {
            "total_queries": len(self.logs),
            "cache_size": len(self.cache),
            "total_cost": f"${cost_report['total_cost']:.6f}",
            "cost_per_analysis": f"${cost_report['cost_per_analysis']:.6f}",
            "budget_remaining": f"${cost_report['budget_remaining']:.6f}",
            "models_used": cost_report['by_model'],
            "by_engagement": cost_report.get('by_engagement', {})
        }


# Test the McKinsey Consulting AI Gateway
gateway = McKinseyAIGateway(budget=0.10)

test_queries = [
    ("What does MECE stand for?", "ENG-2024-0101"),
    ("What does MECE stand for?", "ENG-2024-0205"),  # cached -- different engagement, same query
    ("Compare European vs US fintech competitive landscapes.", "ENG-2024-0205"),
    ("Design a post-merger integration plan for a $3B pharma acquisition.", "ENG-2024-0312"),
    ("What is Porter's Five Forces?", "ENG-2024-0101")
]

for q, eng_id in test_queries:
    result = gateway.query(q, engagement_id=eng_id)
    cached = "CACHED" if result.get('cached') else result.get('complexity', 'n/a')
    cost = result.get('cost', 0)
    print(f"[{cached:8s}] {result['model']:12s} ${cost:.6f} [{eng_id}] | {q[:50]}")

print("\n--- McKinsey Consulting AI Dashboard ---")
for k, v in gateway.get_dashboard().items():
    print(f"  {k}: {v}")

---
## Summary

In this session students learned the operational skills for deploying McKinsey's consulting AI platform to production:

1. **API Design** -- How to structure consulting AI services (engagement assistant, knowledge search, client briefing generator) with validation and error handling.
2. **Semantic Caching** -- How to reduce costs by caching similar consulting queries (recurring market research, industry analyses across engagements).
3. **Monitoring** -- How to track consulting-relevant metrics: response quality for partner review, analysis accuracy, framework coverage.
4. **Model Routing** -- How to route by consulting complexity: simple fact lookups to gpt-4o-mini, complex strategy analyses to gpt-4o.
5. **Cost Tracking** -- How to model consulting AI economics: cost per engagement analysis, tokens per client deliverable, budget alerts.

**Up Next:** After lunch, students choose their capstone track and build a complete McKinsey consulting AI system.